# Chapter 4: Data Transformation Techniques

## Simple aggregations

### How to do it...

In [1]:
import polars as pl

In [2]:
df = pl.read_csv('../data/contoso_sales.csv', try_parse_dates=True)

In [3]:
df.head()

Order Number,Line Number,Order Date,Delivery Date,Customer Name,Customer Gender,Customer Country,Customer Age,Store Name,Product Name,Color,Brand,Category,Subcategory,Quantity,Unit Price,Net Price,Unit Cost,Currency Code,Exchange Rate
i64,i64,date,date,str,str,str,i64,str,str,str,str,str,str,i64,f64,f64,f64,str,f64
284806,1,2017-10-18,2017-10-20,"""Eric Kennedy""","""Male""","""United States""",47,"""Online store""","""Contoso 512MB MP3 Player E51 S…","""Silver""","""Contoso""","""Audio""","""MP4&MP3""",7,11.691,10.288,5.958,"""USD""",1.0
285506,1,2017-10-25,2017-10-26,"""George Tooth""","""Male""","""Australia""",30,"""Online store""","""Contoso 512MB MP3 Player E51 S…","""Silver""","""Contoso""","""Audio""","""MP4&MP3""",1,11.691,11.691,5.958,"""AUD""",1.2967
311002,2,2018-07-07,2018-07-12,"""Caleb Greene""","""Male""","""Australia""",59,"""Online store""","""Contoso 512MB MP3 Player E51 S…","""Silver""","""Contoso""","""Audio""","""MP4&MP3""",6,12.99,12.99,6.62,"""AUD""",1.3484
366307,2,2020-01-11,2020-01-11,"""Isaac Siddins""","""Male""","""Australia""",25,"""Contoso Store Western Australi…","""Contoso 512MB MP3 Player E51 B…","""Blue""","""Contoso""","""Audio""","""MP4&MP3""",4,12.99,12.99,6.62,"""AUD""",1.4545
325708,3,2018-12-01,2018-12-02,"""Mike McQueen""","""Male""","""United States""",56,"""Online store""","""Contoso 512MB MP3 Player E51 B…","""Blue""","""Contoso""","""Audio""","""MP4&MP3""",2,12.99,11.5611,6.62,"""USD""",1.0


In [5]:
df['Color'].unique()

Color
str
"""Yellow"""
"""Gold"""
"""Purple"""
"""Pink"""
"""Grey"""
…
"""Green"""
"""Black"""
"""Silver"""


In [7]:
import polars.selectors as cs

(
    df
    .select(cs.numeric())
    .sum()
)

Order Number,Line Number,Customer Age,Quantity,Unit Price,Net Price,Unit Cost,Exchange Rate
i64,i64,i64,i64,f64,f64,f64,f64
4466019052,16195,725757,43517,4.1785e6,3.9286e6,1.7356e6,14124.4597


In [8]:
s = df.select('Quantity').to_series()
s.sum()

43517

In [11]:
df.select(pl.col('Quantity').sum())

Quantity
i64
43517


In [12]:
df.select(
    pl.col('Customer Name').first().alias('Cust Name First'),
    pl.col('Customer Name').last().alias('Cust Name Last')
)

Cust Name First,Cust Name Last
str,str
"""Eric Kennedy""","""Billy Ratliff"""


In [17]:
df.select(
    pl.col('Customer Name').str.split(' ').list.first().alias('First Name'),
    pl.col('Customer Name').str.split(' ').list.last().alias('Last Name')
)

First Name,Last Name
str,str
"""Eric""","""Kennedy"""
"""George""","""Tooth"""
"""Caleb""","""Greene"""
"""Isaac""","""Siddins"""
"""Mike""","""McQueen"""
…,…
"""Xavier""","""Clay"""
"""Ginger""","""Paine"""
"""Finlay""","""Pugh"""


In [18]:
df.select(cs.numeric()).describe()

statistic,Order Number,Line Number,Customer Age,Quantity,Unit Price,Net Price,Unit Cost,Exchange Rate
str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",13915.0,13915.0,13915.0,13915.0,13915.0,13915.0,13915.0,13915.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",320949.985771,1.163852,52.15645,3.127345,300.28425,282.32739,124.731364,1.015053
"""std""",28431.79136,1.361349,19.133881,2.233597,405.538975,381.738847,147.944094,0.171927
"""min""",269500.0,0.0,19.0,1.0,0.95,0.8265,0.48,0.7015
"""25%""",295902.0,0.0,36.0,1.0,46.99,43.4,21.92,0.8965
"""50%""",319806.0,1.0,52.0,2.0,207.987,194.91,86.68,1.0
"""75%""",345106.0,2.0,68.0,4.0,361.2,336.0,160.93,1.0
"""max""",371503.0,6.0,85.0,10.0,3748.5,3748.5,1241.955,1.5373


### There is more...

In [19]:
df.select(
    (pl.col('Quantity') >= 4).sum()
)

Quantity
u32
4423


In [26]:
df.select(
    pl.col('Quantity').filter(pl.col('Store Name')=='Online store').sum()
)

Quantity
i64
25017


## Using group by aggregations

### How to do it...

In [1]:
import polars as pl

In [2]:
df = pl.read_csv('../data/contoso_sales.csv', try_parse_dates=True)
df.head()

In [27]:
df.group_by('Brand')

In [28]:
(
    df
    .group_by('Brand')
    .agg(pl.col('Quantity').sum().alias('Sum of Quantity'))
    .head()
)

Brand,Sum of Quantity
str,i64
"""A. Datum""",555
"""Northwind Traders""",638
"""Southridge Video""",6749
"""Fabrikam""",1516
"""Contoso""",11132


In [29]:
(
    df
    .group_by('Brand')
    .agg(
        pl.col('Quantity').sum().alias('Sum of Quantity'),
        pl.col('Unit Price').mean().alias('Average Unit Price'),
    )
    .sort('Average Unit Price', descending=True)
    .head()
)

Brand,Sum of Quantity,Average Unit Price
str,i64,f64
"""Fabrikam """,332,795.820192
"""Contoso """,977,715.378268
"""Adventure Works""",4616,620.330242
"""Litware """,161,603.528818
"""Fabrikam""",1516,557.956117


In [31]:
ctx = pl.SQLContext(df=df)

In [35]:
ctx.execute(
    """SELECT Brand,
              SUM(Quantity) AS "Sum of Quantity",
              AVG("Unit Price") AS "Average Unit Price"
       FROM df
       GROUP BY Brand
       ORDER BY "Average Unit Price" DESC
       LIMIT 5
    """
).collect()

Brand,Sum of Quantity,Average Unit Price
str,i64,f64
"""Fabrikam """,332,795.820192
"""Contoso """,977,715.378268
"""Adventure Works""",4616,620.330242
"""Litware """,161,603.528818
"""Fabrikam""",1516,557.956117


In [40]:
(
    df
    .group_by('Brand')
    .agg(
        pl.col('Unit Price').mean().round(2).alias('Average Unit Price'),
        (pl.col('Unit Price').sum() / pl.len()).round(2).alias('Average Unit Price 2'),
        pl.col('Customer Name').first(),
        pl.col('Category').last()
    )
    .sort(['Brand', 'Average Unit Price'], descending=[False, True])
    .head()
)

Brand,Average Unit Price,Average Unit Price 2,Customer Name,Category
str,f64,f64,str,str
"""A. Datum""",280.1,280.1,"""Blažena Salabová""","""Cameras and camcorders """
"""Adventure Works""",620.33,620.33,"""Molly Walters""","""Home Appliances"""
"""Adventure Works """,166.9,166.9,"""James Steinfeld""","""TV and Video"""
"""Contoso""",150.86,150.86,"""Eric Kennedy""","""Cell phones"""
"""Contoso """,715.38,715.38,"""Chiquita Boyd""","""Home Appliances"""


In [36]:
(
    df
    .group_by('Brand')
    .agg(
        pl.col('Unit Price').mean().round(2).alias('Average Unit Price'),
        (pl.col('Unit Price').sum() / pl.len()).round(2).alias('Average Unit Price 2'),
        pl.col('Customer Name').first(),
        pl.col('Category').last()
    )
    .sort('Average Unit Price', descending=True)
    .sort('Brand')
    .head()
)

Brand,Average Unit Price,Average Unit Price 2,Customer Name,Category
str,f64,f64,str,str
"""A. Datum""",280.1,280.1,"""Blažena Salabová""","""Cameras and camcorders """
"""Adventure Works""",620.33,620.33,"""Molly Walters""","""Home Appliances"""
"""Adventure Works """,166.9,166.9,"""James Steinfeld""","""TV and Video"""
"""Contoso""",150.86,150.86,"""Eric Kennedy""","""Cell phones"""
"""Contoso """,715.38,715.38,"""Chiquita Boyd""","""Home Appliances"""


In [43]:
ctx.execute(
    """
        SELECT Brand,
               ROUND(AVG("Unit Price"), 2) AS "Average Unit Price",
               ROUND(SUM("Unit Price") / COUNT("Unit Price"), 2) AS "Average Unit Price 2",
               FIRST("Customer Name"),
               LAST("Category")
        FROM df
        GROUP BY Brand
        ORDER BY Brand, "Average Unit Price" DESC
        LIMIT 5
    """
).collect()

Brand,Average Unit Price,Average Unit Price 2,Customer Name,Category
str,f64,f64,str,str
"""A. Datum""",280.1,280.1,"""Blažena Salabová""","""Cameras and camcorders """
"""Adventure Works""",620.33,620.33,"""Molly Walters""","""Home Appliances"""
"""Adventure Works """,166.9,166.9,"""James Steinfeld""","""TV and Video"""
"""Contoso""",150.86,150.86,"""Eric Kennedy""","""Cell phones"""
"""Contoso """,715.38,715.38,"""Chiquita Boyd""","""Home Appliances"""


In [44]:
(
    pl.scan_csv('../data/contoso_sales.csv', try_parse_dates=True)
    .group_by('Brand')
    .agg(
        pl.col('Unit Price').mean().round(2).alias('Average Unit Price'),
        (pl.col('Unit Price').sum() / pl.len()).round(2).alias('Average Unit Price 2'),
        pl.col('Customer Name').first(),
        pl.col('Category').last()
    )
    .sort('Average Unit Price', descending=True)
    .sort('Brand')
    .collect()
    .head()
)

Brand,Average Unit Price,Average Unit Price 2,Customer Name,Category
str,f64,f64,str,str
"""A. Datum""",280.1,280.1,"""Blažena Salabová""","""Cameras and camcorders """
"""Adventure Works""",620.33,620.33,"""Molly Walters""","""Home Appliances"""
"""Adventure Works """,166.9,166.9,"""James Steinfeld""","""TV and Video"""
"""Contoso""",150.86,150.86,"""Eric Kennedy""","""Cell phones"""
"""Contoso """,715.38,715.38,"""Chiquita Boyd""","""Home Appliances"""


In [51]:
pl.Config.set_fmt_str_lengths = 5
print(df.select('Brand').unique().head(10))

shape: (10, 1)
┌───────────────────┐
│ Brand             │
│ ---               │
│ str               │
╞═══════════════════╡
│ Adventure Works   │
│ Southridge Video  │
│ Northwind Traders │
│ Litware           │
│ Contoso           │
│ Contoso           │
│ Adventure Works   │
│ Proseware         │
│ Litware           │
│ Fabrikam          │
└───────────────────┘


In [54]:
df.select('Brand').unique().head(10)

Brand
str
"""Litware """
"""A. Datum"""
"""Tailspin Toys"""
"""Proseware"""
"""Northwind Traders"""
"""Fabrikam """
"""Contoso """
"""Wide World Importers"""
"""Litware"""


In [55]:
import os
os.environ['POLARS_FMT_STR_LEN'] = str(50)

df.select('Brand').unique().head(10)

Brand
str
"""Litware"""
"""Adventure Works """
"""Adventure Works"""
"""Wide World Importers"""
"""Proseware"""
"""Northwind Traders"""
"""Tailspin Toys"""
"""Fabrikam"""
"""Southridge Video"""


### There is more...

In [56]:
for name, data in df.group_by(['Brand']):
    print(name[0], type(data))

Tailspin Toys <class 'polars.dataframe.frame.DataFrame'>
Litware  <class 'polars.dataframe.frame.DataFrame'>
Proseware <class 'polars.dataframe.frame.DataFrame'>
Wide World Importers <class 'polars.dataframe.frame.DataFrame'>
The Phone Company <class 'polars.dataframe.frame.DataFrame'>
Contoso  <class 'polars.dataframe.frame.DataFrame'>
Northwind Traders <class 'polars.dataframe.frame.DataFrame'>
Adventure Works <class 'polars.dataframe.frame.DataFrame'>
Fabrikam   <class 'polars.dataframe.frame.DataFrame'>
Adventure Works  <class 'polars.dataframe.frame.DataFrame'>
Southridge Video <class 'polars.dataframe.frame.DataFrame'>
Contoso <class 'polars.dataframe.frame.DataFrame'>
Litware <class 'polars.dataframe.frame.DataFrame'>
A. Datum <class 'polars.dataframe.frame.DataFrame'>
Fabrikam <class 'polars.dataframe.frame.DataFrame'>


In [62]:
(
    df
    .group_by('Brand')
    .agg(pl.col('Quantity'))
    .head()
)

Brand,Quantity
str,list[i64]
"""Contoso ""","[3, 1, … 3]"
"""A. Datum""","[2, 1, … 2]"
"""The Phone Company""","[3, 5, … 4]"
"""Fabrikam""","[2, 1, … 2]"
"""Adventure Works""","[5, 7, … 2]"


In [63]:
(
    df
    .group_by('Brand', maintain_order=True)
    .agg(pl.col('Quantity'))
    .head()
)

Brand,Quantity
str,list[i64]
"""Contoso""","[7, 1, … 3]"
"""Wide World Importers""","[2, 8, … 2]"
"""Northwind Traders""","[2, 3, … 2]"
"""Adventure Works ""","[2, 3, … 6]"
"""Adventure Works""","[5, 7, … 2]"


In [64]:
(
    df
    .group_by(
        pl.col('Brand'), 
        'Customer Country',
        pl.col('Order Date').dt.year().alias('Order Year')
        )
    .agg(pl.col('Unit Price').mean())
    .head()
)

Brand,Customer Country,Order Year,Unit Price
str,str,i32,f64
"""Proseware""","""Australia""",2020,1299.0
"""The Phone Company""","""United States""",2019,284.531157
"""Litware""","""Canada""",2020,369.663333
"""Wide World Importers""","""France""",2020,194.2475
"""A. Datum""","""Canada""",2020,243.45


## Aggregating values across multiple columns

### How to do it...

In [6]:
import polars as pl

In [65]:
df = pl.read_csv('../data/pokemon.csv')
df.head()

#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
i64,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,bool
1,"""Bulbasaur""","""Grass""","""Poison""",318,45,49,49,65,65,45,1,false
2,"""Ivysaur""","""Grass""","""Poison""",405,60,62,63,80,80,60,1,false
3,"""Venusaur""","""Grass""","""Poison""",525,80,82,83,100,100,80,1,false
3,"""VenusaurMega Venusaur""","""Grass""","""Poison""",625,80,100,123,122,120,80,1,false
4,"""Charmander""","""Fire""",null,309,39,52,43,60,50,65,1,false


In [66]:
(
    df
    .select('HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed')
    .sum_horizontal().alias('Total 2')
    .head()
)

Total 2
i64
318
405
525
625
309
405
534
634
634


In [67]:
(
    df
    .with_columns(
        pl.sum_horizontal('HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed').alias('Total 2')
    )
    .head()
)

#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary,Total 2
i64,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,bool,i64
1,"""Bulbasaur""","""Grass""","""Poison""",318,45,49,49,65,65,45,1,false,318
2,"""Ivysaur""","""Grass""","""Poison""",405,60,62,63,80,80,60,1,false,405
3,"""Venusaur""","""Grass""","""Poison""",525,80,82,83,100,100,80,1,false,525
3,"""VenusaurMega Venusaur""","""Grass""","""Poison""",625,80,100,123,122,120,80,1,false,625
4,"""Charmander""","""Fire""",null,309,39,52,43,60,50,65,1,false,309


In [68]:
(
    df
    .with_columns(
        pl.concat_list('HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed').list.sum().alias('Total 2')
    )
    .head()
)

#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary,Total 2
i64,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,bool,i64
1,"""Bulbasaur""","""Grass""","""Poison""",318,45,49,49,65,65,45,1,false,318
2,"""Ivysaur""","""Grass""","""Poison""",405,60,62,63,80,80,60,1,false,405
3,"""Venusaur""","""Grass""","""Poison""",525,80,82,83,100,100,80,1,false,525
3,"""VenusaurMega Venusaur""","""Grass""","""Poison""",625,80,100,123,122,120,80,1,false,625
4,"""Charmander""","""Fire""",null,309,39,52,43,60,50,65,1,false,309


In [70]:
(
    df.with_columns(
        pl.concat_arr('HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed').arr.sum().alias('Total 2')
    )
    .head()
)

#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary,Total 2
i64,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,bool,i64
1,"""Bulbasaur""","""Grass""","""Poison""",318,45,49,49,65,65,45,1,false,318
2,"""Ivysaur""","""Grass""","""Poison""",405,60,62,63,80,80,60,1,false,405
3,"""Venusaur""","""Grass""","""Poison""",525,80,82,83,100,100,80,1,false,525
3,"""VenusaurMega Venusaur""","""Grass""","""Poison""",625,80,100,123,122,120,80,1,false,625
4,"""Charmander""","""Fire""",null,309,39,52,43,60,50,65,1,false,309


In [71]:
cols = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
(
    df
    .with_columns(
        pl.reduce(
            function=lambda acc, col: acc + col, 
            exprs=pl.col(cols)
        )
        .alias('Total 2')
    )
    .head()
)

#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary,Total 2
i64,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,bool,i64
1,"""Bulbasaur""","""Grass""","""Poison""",318,45,49,49,65,65,45,1,false,318
2,"""Ivysaur""","""Grass""","""Poison""",405,60,62,63,80,80,60,1,false,405
3,"""Venusaur""","""Grass""","""Poison""",525,80,82,83,100,100,80,1,false,525
3,"""VenusaurMega Venusaur""","""Grass""","""Poison""",625,80,100,123,122,120,80,1,false,625
4,"""Charmander""","""Fire""",null,309,39,52,43,60,50,65,1,false,309


In [80]:
(
    df
    .with_columns(
        pl.fold(
            acc=pl.lit(1, dtype=pl.Int64), 
            function=lambda acc, col: acc * col, 
            exprs=pl.col(cols)
        )
        .alias('Product')
    )
    .head()
)

#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary,Product
i64,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,bool,i64
1,"""Bulbasaur""","""Grass""","""Poison""",318,45,49,49,65,65,45,1,false,20542055625
2,"""Ivysaur""","""Grass""","""Poison""",405,60,62,63,80,80,60,1,false,89994240000
3,"""Venusaur""","""Grass""","""Poison""",525,80,82,83,100,100,80,1,false,435584000000
3,"""VenusaurMega Venusaur""","""Grass""","""Poison""",625,80,100,123,122,120,80,1,false,1152460800000
4,"""Charmander""","""Fire""",null,309,39,52,43,60,50,65,1,false,17004780000


### There is more...

In [81]:
(
    df
    .filter(
        pl.fold(
            acc=pl.lit(True), 
            function=lambda acc, col: acc & col, 
            exprs=pl.col(cols) > 80
        )
    )
    .head()
)

#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
i64,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,bool
144,"""Articuno""","""Ice""","""Flying""",580,90,85,100,95,125,85,1,true
145,"""Zapdos""","""Electric""","""Flying""",580,90,90,85,125,90,100,1,true
146,"""Moltres""","""Fire""","""Flying""",580,90,100,90,125,85,90,1,true
150,"""Mewtwo""","""Psychic""",null,680,106,110,90,154,90,130,1,true


In [85]:
df.filter(
    pl.all_horizontal(pl.col(cols) > 80)
)

#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
i64,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,bool
144,"""Articuno""","""Ice""","""Flying""",580,90,85,100,95,125,85,1,true
145,"""Zapdos""","""Electric""","""Flying""",580,90,90,85,125,90,100,1,true
146,"""Moltres""","""Fire""","""Flying""",580,90,100,90,125,85,90,1,true
150,"""Mewtwo""","""Psychic""",null,680,106,110,90,154,90,130,1,true


In [86]:
(
    df
    .filter(
        pl.all_horizontal(pl.col(cols) > 80)
    )
    .head()
)

#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
i64,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,bool
144,"""Articuno""","""Ice""","""Flying""",580,90,85,100,95,125,85,1,true
145,"""Zapdos""","""Electric""","""Flying""",580,90,90,85,125,90,100,1,true
146,"""Moltres""","""Fire""","""Flying""",580,90,100,90,125,85,90,1,true
150,"""Mewtwo""","""Psychic""",null,680,106,110,90,154,90,130,1,true


In [90]:
str_cols = ['Name', 'Type 1', 'Type 2']
str_combined = pl.fold(acc=pl.lit(''), function=lambda acc, col: acc + col, exprs=str_cols).alias('Str Combined')
str_cols.append(str_combined)
df.select(str_cols).head()

Name,Type 1,Type 2,Str Combined
str,str,str,str
"""Bulbasaur""","""Grass""","""Poison""","""BulbasaurGrassPoison"""
"""Ivysaur""","""Grass""","""Poison""","""IvysaurGrassPoison"""
"""Venusaur""","""Grass""","""Poison""","""VenusaurGrassPoison"""
"""VenusaurMega Venusaur""","""Grass""","""Poison""","""VenusaurMega VenusaurGrassPoison"""
"""Charmander""","""Fire""",null,null


In [93]:
str_cols = ['Name', 'Type 1', 'Type 2']
df.select(pl.concat_str(str_cols, separator=' ', ignore_nulls=True)).head()

Name
str
"""Bulbasaur Grass Poison"""
"""Ivysaur Grass Poison"""
"""Venusaur Grass Poison"""
"""VenusaurMega Venusaur Grass Poison"""
"""Charmander Fire"""


## Computing over groups with window functions

### How to do it...

In [ ]:
import polars as pl

In [94]:
import os 
os.environ['POLARS_FMT_STR_LEN'] = str(50) 

In [95]:
df = pl.read_csv('../data/contoso_sales.csv', try_parse_dates=True)
df = df.with_columns(
    (pl.col('Quantity') * pl.col('Net Price')).round(2).alias('Sales Amount')
)
df.head()

Order Number,Line Number,Order Date,Delivery Date,Customer Name,Customer Gender,Customer Country,Customer Age,Store Name,Product Name,Color,Brand,Category,Subcategory,Quantity,Unit Price,Net Price,Unit Cost,Currency Code,Exchange Rate,Sales Amount
i64,i64,date,date,str,str,str,i64,str,str,str,str,str,str,i64,f64,f64,f64,str,f64,f64
284806,1,2017-10-18,2017-10-20,"""Eric Kennedy""","""Male""","""United States""",47,"""Online store""","""Contoso 512MB MP3 Player E51 Silver""","""Silver""","""Contoso""","""Audio""","""MP4&MP3""",7,11.691,10.288,5.958,"""USD""",1.0,72.02
285506,1,2017-10-25,2017-10-26,"""George Tooth""","""Male""","""Australia""",30,"""Online store""","""Contoso 512MB MP3 Player E51 Silver""","""Silver""","""Contoso""","""Audio""","""MP4&MP3""",1,11.691,11.691,5.958,"""AUD""",1.2967,11.69
311002,2,2018-07-07,2018-07-12,"""Caleb Greene""","""Male""","""Australia""",59,"""Online store""","""Contoso 512MB MP3 Player E51 Silver""","""Silver""","""Contoso""","""Audio""","""MP4&MP3""",6,12.99,12.99,6.62,"""AUD""",1.3484,77.94
366307,2,2020-01-11,2020-01-11,"""Isaac Siddins""","""Male""","""Australia""",25,"""Contoso Store Western Australia""","""Contoso 512MB MP3 Player E51 Blue""","""Blue""","""Contoso""","""Audio""","""MP4&MP3""",4,12.99,12.99,6.62,"""AUD""",1.4545,51.96
325708,3,2018-12-01,2018-12-02,"""Mike McQueen""","""Male""","""United States""",56,"""Online store""","""Contoso 512MB MP3 Player E51 Blue""","""Blue""","""Contoso""","""Audio""","""MP4&MP3""",2,12.99,11.5611,6.62,"""USD""",1.0,23.12


In [96]:
sales_by_cat = df.select(
    'Category',
    'Subcategory',
    pl.col('Sales Amount').sum().over('Category').alias('Sales Amt per Cat')
)
sales_by_cat.head()    

Category,Subcategory,Sales Amt per Cat
str,str,f64
"""Audio""","""MP4&MP3""",238355.95
"""Audio""","""MP4&MP3""",238355.95
"""Audio""","""MP4&MP3""",238355.95
"""Audio""","""MP4&MP3""",238355.95
"""Audio""","""MP4&MP3""",238355.95


In [97]:
ctx = pl.SQLContext(df=df)

ctx.execute(
    """SELECT Category, Subcategory,
              SUM("Sales Amount") OVER (PARTITION BY Category) AS "Sales Amt per Cat"
       FROM df
       LIMIT 5
    """
).collect()

Category,Subcategory,Sales Amt per Cat
str,str,f64
"""Audio""","""MP4&MP3""",238355.95
"""Audio""","""MP4&MP3""",238355.95
"""Audio""","""MP4&MP3""",238355.95
"""Audio""","""MP4&MP3""",238355.95
"""Audio""","""MP4&MP3""",238355.95


In [98]:
pl.sql(
    """SELECT Category, Subcategory,
              SUM("Sales Amount") OVER (PARTITION BY Category) AS "Sales Amt per Cat"
       FROM df
       LIMIT 5
    """
).collect()

Category,Subcategory,Sales Amt per Cat
str,str,f64
"""Audio""","""MP4&MP3""",238355.95
"""Audio""","""MP4&MP3""",238355.95
"""Audio""","""MP4&MP3""",238355.95
"""Audio""","""MP4&MP3""",238355.95
"""Audio""","""MP4&MP3""",238355.95


In [99]:
sales_by_cat.filter(pl.col('Category')=='Audio').unique().head()

Category,Subcategory,Sales Amt per Cat
str,str,f64
"""Audio""","""Bluetooth Headphones""",238355.95
"""Audio""","""MP4&MP3""",238355.95
"""Audio""","""Recording Pen""",238355.95


In [102]:
df.select(pl.when(pl.col("Sales Amount") > 1000).then(pl.lit('High')).otherwise(pl.lit("Low")))

literal
str
"""Low"""
"""Low"""
"""Low"""
"""Low"""
"""Low"""
…
"""Low"""
"""Low"""
"""Low"""


In [103]:
df.shape, sales_by_cat.shape 

((13915, 21), (13915, 3))

In [ ]:
df.sql(
    f"""
    SELECT DISTINCT Category, Brand, Subcategory,
           AVG("Sales Amount") OVER (PARTITION BY Category, Brand) AS "Avg Sales Amt per Cat and Brand"
    FROM SELF
    WHERE Category = 'Computers'
    ORDER BY Brand
    LIMIT 10
    """
)

Category,Brand,Subcategory,Avg Sales Amt per Cat and Brand
str,str,str,f64
"""Computers""","""Adventure Works""","""Laptops""",1797.371618
"""Computers""","""Adventure Works""","""Desktops""",1797.371618
"""Computers""","""Adventure Works""","""Monitors""",1797.371618
"""Computers""","""Contoso""","""Projectors & Screens""",689.986461
"""Computers""","""Contoso""","""Computers Accessories""",689.986461
"""Computers""","""Fabrikam""","""Laptops""",1982.065906
"""Computers""","""Proseware""","""Laptops""",1095.304828
"""Computers""","""Proseware""","""Monitors""",1095.304828
"""Computers""","""Proseware""","""Projectors & Screens""",1095.304828


In [104]:
(
    df
    .select(
        'Category',
        'Brand',
        'Subcategory',
        pl.col('Sales Amount').mean().over('Category', 'Brand').alias('Avg Sales per Cat and Brand')
    )
    .filter(
        (pl.col('Category')=='Computers') 
    )
    .unique()
    .sort('Brand')
    .head(10)
)

Category,Brand,Subcategory,Avg Sales per Cat and Brand
str,str,str,f64
"""Computers""","""Adventure Works""","""Monitors""",1797.371618
"""Computers""","""Adventure Works""","""Laptops""",1797.371618
"""Computers""","""Adventure Works""","""Desktops""",1797.371618
"""Computers""","""Contoso""","""Computers Accessories""",689.986461
"""Computers""","""Contoso""","""Projectors & Screens""",689.986461
"""Computers""","""Fabrikam""","""Laptops""",1982.065906
"""Computers""","""Proseware""","""Printers, Scanners & Fax""",1095.304828
"""Computers""","""Proseware""","""Monitors""",1095.304828
"""Computers""","""Proseware""","""Projectors & Screens""",1095.304828


In [110]:
from datetime import date

curr_yr = date.today().year
cust_birth_yr = curr_yr - pl.col('Customer Age')

(
    df
    .select(
        'Category',
        'Brand',
        'Customer Age',
        pl.col('Sales Amount').mean().over('Category', cust_birth_yr).alias('Avg Sales per Cat') 
    )
    .filter(pl.col('Category')=='Computers')
    .unique()
    .sort('Customer Age')
    .head(10)
)

Category,Brand,Customer Age,Avg Sales per Cat
str,str,i64,f64
"""Computers""","""Southridge Video""",19,1665.993158
"""Computers""","""Wide World Importers""",19,1665.993158
"""Computers""","""Contoso""",19,1665.993158
"""Computers""","""Proseware""",19,1665.993158
"""Computers""","""Fabrikam""",19,1665.993158
"""Computers""","""Adventure Works""",19,1665.993158
"""Computers""","""Adventure Works""",20,2094.541406
"""Computers""","""Fabrikam""",20,2094.541406
"""Computers""","""Proseware""",20,2094.541406


In [111]:
curr_yr = date.today().year
ctx = pl.SQLContext(df=df)

result = ctx.execute(
    f"""
    SELECT DISTINCT
           Category,
           Brand,
           "Customer Age",
           AVG("Sales Amount") OVER (
               PARTITION BY Category, ({curr_yr} - "Customer Age")
           ) AS "Avg Sales per Cat"
    FROM df
    WHERE Category = 'Computers'
    ORDER BY "Customer Age"
    LIMIT 10
    """
).collect()

result

Category,Brand,Customer Age,Avg Sales per Cat
str,str,i64,f64
"""Computers""","""Fabrikam""",19,1665.993158
"""Computers""","""Adventure Works""",19,1665.993158
"""Computers""","""Wide World Importers""",19,1665.993158
"""Computers""","""Proseware""",19,1665.993158
"""Computers""","""Contoso""",19,1665.993158
"""Computers""","""Southridge Video""",19,1665.993158
"""Computers""","""Fabrikam""",20,2094.541406
"""Computers""","""Adventure Works""",20,2094.541406
"""Computers""","""Wide World Importers""",20,2094.541406


In [112]:
(
    df
    .group_by('Category')
    .agg(pl.col('Sales Amount').max().alias('Max Sales Amt'))
    .with_columns(
        pl.col('Max Sales Amt').rank(descending=True).alias('Rank')
    )
    .sort('Rank')
)

Category,Max Sales Amt,Rank
str,f64,f64
"""TV and Video""",28999.9,1.0
"""Home Appliances""",28479.91,2.0
"""Computers""",19992.0,3.0
"""Cameras and camcorders """,10810.8,4.0
"""Cell phones""",5183.2,5.0
"""Music, Movies and Audio Books""",3041.88,6.0
"""Audio""",2871.2,7.0
"""Games and Toys""",2813.16,8.0


In [115]:
ctx.execute(
    f"""
    WITH cat_max AS (
        SELECT Category,
               MAX("Sales Amount") AS "Max Sales Amt"
        FROM df
        GROUP BY Category
    )
    SELECT Category,
        "Max Sales Amt",
        CAST(
            RANK() OVER (
                ORDER BY "Max Sales Amt" DESC
            ) + (
                COUNT(*) OVER (PARTITION BY "Max Sales Amt") - 1
            ) / 2.0 AS DOUBLE
        ) AS "Rank"
    FROM cat_max
    ORDER BY "Rank"
    """
).collect()

Category,Max Sales Amt,Rank
str,f64,f64
"""TV and Video""",28999.9,1.0
"""Home Appliances""",28479.91,2.0
"""Computers""",19992.0,3.0
"""Cameras and camcorders """,10810.8,4.0
"""Cell phones""",5183.2,5.0
"""Music, Movies and Audio Books""",3041.88,6.0
"""Audio""",2871.2,7.0
"""Games and Toys""",2813.16,8.0


In [116]:
(
    df
    .group_by('Category', 'Subcategory')
    .agg(pl.col('Sales Amount').max().round().cast(pl.Int64).alias('Max Sales Amt'))
    .with_columns(
        pl.col('Max Sales Amt').rank(descending=True).over('Category').cast(pl.Int64).alias('Rank')
    )
    .filter(pl.col('Category').is_in(['Audio', 'Computers']))
    .sort(['Category', 'Rank'])
)

Category,Subcategory,Max Sales Amt,Rank
str,str,i64,i64
"""Audio""","""Recording Pen""",2871,1
"""Audio""","""Bluetooth Headphones""",2250,2
"""Audio""","""MP4&MP3""",2095,3
"""Computers""","""Projectors & Screens""",19992,1
"""Computers""","""Laptops""",19485,2
"""Computers""","""Desktops""",14535,3
"""Computers""","""Monitors""",11425,4
"""Computers""","""Printers, Scanners & Fax""",2508,5
"""Computers""","""Computers Accessories""",2424,6


In [ ]:
ctx = pl.SQLContext(df=df)

result = ctx.execute(
    """
    WITH subcat_max AS (
        SELECT Category,
               Subcategory,
               CAST(
                   ROUND(MAX("Sales Amount"), 0) AS BIGINT
               ) AS "Max Sales Amt"
        FROM df
        WHERE Category IN ('Audio', 'Computers')
        GROUP BY Category, Subcategory
    )
    SELECT Category,
        Subcategory,
        "Max Sales Amt",
        CAST(
            RANK() OVER (
                PARTITION BY Category
                ORDER BY "Max Sales Amt" DESC
            )
            + (
                COUNT(*) OVER (
                    PARTITION BY Category, "Max Sales Amt"
                ) - 1
            ) / 2.0
            AS BIGINT
        ) AS "Rank"
    FROM subcat_max
    ORDER BY Category, "Rank"
    """
).collect()
result

Category,Subcategory,Max Sales Amt,Rank
str,str,i64,i64
"""Audio""","""Recording Pen""",2871,1
"""Audio""","""Bluetooth Headphones""",2250,2
"""Audio""","""MP4&MP3""",2095,3
"""Computers""","""Projectors & Screens""",19992,1
"""Computers""","""Laptops""",19485,2
"""Computers""","""Desktops""",14535,3
"""Computers""","""Monitors""",11425,4
"""Computers""","""Printers, Scanners & Fax""",2508,5
"""Computers""","""Computers Accessories""",2424,6


### There is more...

In [126]:
max_sales_rank = (
    df
    .group_by('Category', 'Subcategory')
    .agg(pl.col('Sales Amount').max().round().cast(pl.Int64).alias('Max Sales Amt'))
    .with_columns(
        pl.col('Max Sales Amt').rank(descending=True).over('Category').cast(pl.Int64).alias('Rank')
    )
    .filter(pl.col('Category').is_in(['Audio', 'Computers']))
    .sort(['Category', 'Rank'])
)

In [127]:
max_sales_rank.with_columns(
    pl.col('Subcategory')
    .sort_by('Max Sales Amt')
    .head(3)
    .over('Category', mapping_strategy='join')
    .alias('Lowest 3 Subcat per Cat')
)

Category,Subcategory,Max Sales Amt,Rank,Lowest 3 Subcat per Cat
str,str,i64,i64,list[str]
"""Audio""","""Recording Pen""",2871,1,"[""MP4&MP3"", ""Bluetooth Headphones"", ""Recording Pen""]"
"""Audio""","""Bluetooth Headphones""",2250,2,"[""MP4&MP3"", ""Bluetooth Headphones"", ""Recording Pen""]"
"""Audio""","""MP4&MP3""",2095,3,"[""MP4&MP3"", ""Bluetooth Headphones"", ""Recording Pen""]"
"""Computers""","""Projectors & Screens""",19992,1,"[""Computers Accessories"", ""Printers, Scanners & Fax"", ""Monitors""]"
"""Computers""","""Laptops""",19485,2,"[""Computers Accessories"", ""Printers, Scanners & Fax"", ""Monitors""]"
"""Computers""","""Desktops""",14535,3,"[""Computers Accessories"", ""Printers, Scanners & Fax"", ""Monitors""]"
"""Computers""","""Monitors""",11425,4,"[""Computers Accessories"", ""Printers, Scanners & Fax"", ""Monitors""]"
"""Computers""","""Printers, Scanners & Fax""",2508,5,"[""Computers Accessories"", ""Printers, Scanners & Fax"", ""Monitors""]"
"""Computers""","""Computers Accessories""",2424,6,"[""Computers Accessories"", ""Printers, Scanners & Fax"", ""Monitors""]"


In [125]:
max_sales_rank.with_columns(
    pl.col('Subcategory')
    .sort_by('Max Sales Amt')
    .over('Category', mapping_strategy='explode')
    .alias('Subcategory Sorted by Max Sales Amt Ascending')
)

Category,Subcategory,Max Sales Amt,Rank,Subcategory Sorted by Max Sales Amt Ascending
str,str,i64,i64,str
"""Audio""","""Recording Pen""",2871,1,"""MP4&MP3"""
"""Audio""","""Bluetooth Headphones""",2250,2,"""Bluetooth Headphones"""
"""Audio""","""MP4&MP3""",2095,3,"""Recording Pen"""
"""Computers""","""Projectors & Screens""",19992,1,"""Computers Accessories"""
"""Computers""","""Laptops""",19485,2,"""Printers, Scanners & Fax"""
"""Computers""","""Desktops""",14535,3,"""Monitors"""
"""Computers""","""Monitors""",11425,4,"""Desktops"""
"""Computers""","""Printers, Scanners & Fax""",2508,5,"""Laptops"""
"""Computers""","""Computers Accessories""",2424,6,"""Projectors & Screens"""


In [130]:
(
    max_sales_rank
    .sort('Subcategory')
    .with_columns(
        pl.col('Subcategory')
        .sort_by('Max Sales Amt')
        .over('Category', mapping_strategy='explode')
        .alias('Subcategory Sorted by Max Sales Amt Ascending')
    )
)

Category,Subcategory,Max Sales Amt,Rank,Subcategory Sorted by Max Sales Amt Ascending
str,str,i64,i64,str
"""Audio""","""Bluetooth Headphones""",2250,2,"""MP4&MP3"""
"""Computers""","""Computers Accessories""",2424,6,"""Bluetooth Headphones"""
"""Computers""","""Desktops""",14535,3,"""Recording Pen"""
"""Computers""","""Laptops""",19485,2,"""Computers Accessories"""
"""Audio""","""MP4&MP3""",2095,3,"""Printers, Scanners & Fax"""
"""Computers""","""Monitors""",11425,4,"""Monitors"""
"""Computers""","""Printers, Scanners & Fax""",2508,5,"""Desktops"""
"""Computers""","""Projectors & Screens""",19992,1,"""Laptops"""
"""Audio""","""Recording Pen""",2871,1,"""Projectors & Screens"""


## Applying UDFs

### How to do it...

In [10]:
import polars as pl

In [131]:
df = pl.read_csv('../data/contoso_sales.csv', try_parse_dates=True)
df.head()

Order Number,Line Number,Order Date,Delivery Date,Customer Name,Customer Gender,Customer Country,Customer Age,Store Name,Product Name,Color,Brand,Category,Subcategory,Quantity,Unit Price,Net Price,Unit Cost,Currency Code,Exchange Rate
i64,i64,date,date,str,str,str,i64,str,str,str,str,str,str,i64,f64,f64,f64,str,f64
284806,1,2017-10-18,2017-10-20,"""Eric Kennedy""","""Male""","""United States""",47,"""Online store""","""Contoso 512MB MP3 Player E51 Silver""","""Silver""","""Contoso""","""Audio""","""MP4&MP3""",7,11.691,10.288,5.958,"""USD""",1.0
285506,1,2017-10-25,2017-10-26,"""George Tooth""","""Male""","""Australia""",30,"""Online store""","""Contoso 512MB MP3 Player E51 Silver""","""Silver""","""Contoso""","""Audio""","""MP4&MP3""",1,11.691,11.691,5.958,"""AUD""",1.2967
311002,2,2018-07-07,2018-07-12,"""Caleb Greene""","""Male""","""Australia""",59,"""Online store""","""Contoso 512MB MP3 Player E51 Silver""","""Silver""","""Contoso""","""Audio""","""MP4&MP3""",6,12.99,12.99,6.62,"""AUD""",1.3484
366307,2,2020-01-11,2020-01-11,"""Isaac Siddins""","""Male""","""Australia""",25,"""Contoso Store Western Australia""","""Contoso 512MB MP3 Player E51 Blue""","""Blue""","""Contoso""","""Audio""","""MP4&MP3""",4,12.99,12.99,6.62,"""AUD""",1.4545
325708,3,2018-12-01,2018-12-02,"""Mike McQueen""","""Male""","""United States""",56,"""Online store""","""Contoso 512MB MP3 Player E51 Blue""","""Blue""","""Contoso""","""Audio""","""MP4&MP3""",2,12.99,11.5611,6.62,"""USD""",1.0


In [132]:
def get_first_name(full_name: str) -> str:
    return full_name.split(' ')[0]

In [139]:
%%timeit 1
df.select(
    'Customer Name',
    pl.col('Customer Name').map_elements(lambda el: get_first_name(el), return_dtype=pl.String).alias('Customer First Name')
).head()

6.33 ms ± 130 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [140]:
%%timeit 1
df.select(
    'Customer Name',
    pl.col('Customer Name').str.split(' ').list.first().alias('Customer First Name')
).head()

570 μs ± 13.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [142]:
%%timeit 1
df.select(
    'Customer Name',
    pl.col('Customer Name').map_elements(lambda el: el.split(' ')[0], return_dtype=pl.String).alias('Customer First Name')
).head()

5.68 ms ± 144 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [146]:
def age_to_range(age: int) -> str:
    if age < 18:
        return '~17'
    elif age <= 30:
        return '18~30'
    elif age <= 50:
        return '31~50'
    elif age <= 70:
        return '51~70'
    else:
        return '71~'

In [147]:
%%timeit 1
df.select(
    'Customer Age',
    pl.col('Customer Age').map_elements(lambda el: age_to_range(el), return_dtype=pl.String).alias('Age Range')
).head()

3.65 ms ± 73 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [145]:
%%timeit 1
age = pl.col("Customer Age")

df.select(
    "Customer Age",
    pl.when(age.is_null())
    .then(pl.lit(None, dtype=pl.String))
    .when(age < 18)
    .then(pl.lit("~17"))
    .when(age <= 30)
    .then(pl.lit("18~30"))
    .when(age <= 50)
    .then(pl.lit("31~50"))
    .when(age <= 70)
    .then(pl.lit("51~70"))
    .otherwise(pl.lit("71~"))
    .alias("Age Range"),
).head()

440 μs ± 25.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [151]:
%%timeit 1
def ages_to_ranges(ages: pl.Series) -> pl.Series:
    return ages.to_frame("age").select(
        pl.when(pl.col("age").is_null())
        .then(pl.lit(None, dtype=pl.String))
        .when(pl.col("age") < 18)
        .then(pl.lit("~17"))
        .when(pl.col("age") <= 30)
        .then(pl.lit("18~30"))
        .when(pl.col("age") <= 50)
        .then(pl.lit("31~50"))
        .when(pl.col("age") <= 70)
        .then(pl.lit("50~70"))
        .otherwise(pl.lit("71~"))
        .alias("Age Range")
    ).to_series()


df.select(
    "Customer Age",
    pl.col("Customer Age")
    .map_batches(
        ages_to_ranges,
        return_dtype=pl.String,
        is_elementwise=True,
    )
    .alias("Age Range"),
).head()

1.74 ms ± 30.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### There is more...

In [152]:
df.select(
    'Customer Name',
    pl.col('Customer Name').str.split(' ').list.first().alias('Customer First Name')
).head()

Customer Name,Customer First Name
str,str
"""Eric Kennedy""","""Eric"""
"""George Tooth""","""George"""
"""Caleb Greene""","""Caleb"""
"""Isaac Siddins""","""Isaac"""
"""Mike McQueen""","""Mike"""


In [153]:
df.select(
    'Customer Age',
    pl.when(pl.col('Customer Age')<18).then(pl.lit('~17'))
    .when(pl.col('Customer Age')<=30).then(pl.lit('18~30'))
    .when(pl.col('Customer Age')<=50).then(pl.lit('31~50'))
    .when(pl.col('Customer Age')<=70).then(pl.lit('51~70'))
    .when(pl.col('Customer Age')>70).then(pl.lit('71~'))
    .alias('Age Range')
).head()

Customer Age,Age Range
i64,str
47,"""31~50"""
30,"""18~30"""
59,"""51~70"""
25,"""18~30"""
56,"""51~70"""


In [ ]:
%%timeit
df.select(
    'Customer Name',
    pl.col('Customer Name').map_elements(lambda el: el.split(' ')[0], return_dtype=pl.String).alias('Customer First Name')
).head()

In [ ]:
%%timeit
df.select(
    'Customer Name',
    pl.col('Customer Name').str.split(' ').list.first().alias('Customer First Name')
).head()

## Using SQL for data transformations

In [154]:
import polars as pl
df = pl.read_csv('../data/contoso_sales.csv', try_parse_dates=True)
df.head()

Order Number,Line Number,Order Date,Delivery Date,Customer Name,Customer Gender,Customer Country,Customer Age,Store Name,Product Name,Color,Brand,Category,Subcategory,Quantity,Unit Price,Net Price,Unit Cost,Currency Code,Exchange Rate
i64,i64,date,date,str,str,str,i64,str,str,str,str,str,str,i64,f64,f64,f64,str,f64
284806,1,2017-10-18,2017-10-20,"""Eric Kennedy""","""Male""","""United States""",47,"""Online store""","""Contoso 512MB MP3 Player E51 Silver""","""Silver""","""Contoso""","""Audio""","""MP4&MP3""",7,11.691,10.288,5.958,"""USD""",1.0
285506,1,2017-10-25,2017-10-26,"""George Tooth""","""Male""","""Australia""",30,"""Online store""","""Contoso 512MB MP3 Player E51 Silver""","""Silver""","""Contoso""","""Audio""","""MP4&MP3""",1,11.691,11.691,5.958,"""AUD""",1.2967
311002,2,2018-07-07,2018-07-12,"""Caleb Greene""","""Male""","""Australia""",59,"""Online store""","""Contoso 512MB MP3 Player E51 Silver""","""Silver""","""Contoso""","""Audio""","""MP4&MP3""",6,12.99,12.99,6.62,"""AUD""",1.3484
366307,2,2020-01-11,2020-01-11,"""Isaac Siddins""","""Male""","""Australia""",25,"""Contoso Store Western Australia""","""Contoso 512MB MP3 Player E51 Blue""","""Blue""","""Contoso""","""Audio""","""MP4&MP3""",4,12.99,12.99,6.62,"""AUD""",1.4545
325708,3,2018-12-01,2018-12-02,"""Mike McQueen""","""Male""","""United States""",56,"""Online store""","""Contoso 512MB MP3 Player E51 Blue""","""Blue""","""Contoso""","""Audio""","""MP4&MP3""",2,12.99,11.5611,6.62,"""USD""",1.0


### How to do it...

In [155]:
ctx = pl.SQLContext(eager=True)
ctx.register('df', df)
ctx.execute(
    """
      select
        `Customer Name`,
        Brand,
        Category
      from df limit 5
    """
)


Customer Name,Brand,Category
str,str,str
"""Eric Kennedy""","""Contoso""","""Audio"""
"""George Tooth""","""Contoso""","""Audio"""
"""Caleb Greene""","""Contoso""","""Audio"""
"""Isaac Siddins""","""Contoso""","""Audio"""
"""Mike McQueen""","""Contoso""","""Audio"""


In [156]:
ctx.execute(
    """
      select
        Brand,
        avg(Quantity) as `Avg Quantity` 
      from df
      group by 
        Brand
      order by 
        `Avg Quantity` desc
      limit 5
    """
)

Brand,Avg Quantity
str,f64
"""Fabrikam""",3.225532
"""Northwind Traders""",3.222222
"""Wide World Importers""",3.193811
"""Fabrikam """,3.192308
"""Southridge Video""",3.189509


In [37]:
pl.SQLContext(lf=df.lazy()).execute(
    """
        select 
            Brand,
            Category
        from lf
        limit 5
    """
).collect()

Brand,Category
str,str
"""Contoso""","""Audio"""
"""Contoso""","""Audio"""
"""Contoso""","""Audio"""
"""Contoso""","""Audio"""
"""Contoso""","""Audio"""
